In [ ]:
import math
import heapq
import random

# -------------------------
# Customers Coordinates
# -------------------------
customers = {0:(1,1),1:(3,5),2:(5,2),3:(2,4),4:(4,0)}

# -------------------------
# Distance Function
# -------------------------
def distance(a,b):
    x1,y1 = a
    x2,y2 = b
    return math.sqrt((x1-x2)**2 + (y1-y2)**2)

# -------------------------
# -------------------------
# A* Algorithm on Grid
# -------------------------
def astar(grid, start, goal):
    rows, cols = len(grid), len(grid[0])
    open_set = []
    heapq.heappush(open_set, (0+heuristic(start, goal), 0, start, [start]))
    visited = set()

    while open_set:
        f, g, current, path = heapq.heappop(open_set)
        if current == goal:
            return path, g

        if current in visited:
            continue
        visited.add(current)

        x, y = current
        for dx, dy in [(-1,0),(1,0),(0,-1),(0,1)]:
            nx, ny = x+dx, y+dy
            if 0<=nx<rows and 0<=ny<cols:
                cost = grid[nx][ny]
                if (nx,ny) not in visited:
                    heapq.heappush(open_set,(g+cost+heuristic((nx,ny),goal), g+cost, (nx,ny), path+[(nx,ny)]))
    return None, math.inf

def heuristic(a,b):
    # Euclidean distance
    return math.sqrt((a[0]-b[0])**2 + (a[1]-b[1])**2)

# -------------------------
# Genetic Algorithm for Delivery Route
# -------------------------
def total_distance(route):
    total = 0
    for i in range(len(route)-1):
        total += distance(route[i], route[i+1])
    total += distance(route[-1], route[0])  # return to start
    return total

def init_population(size, nodes):
    population = []
    for _ in range(size):
        ind = nodes[:]
        random.shuffle(ind)
        population.append(ind)
    return population

def tournament_selection(pop, k=3):
    selected = random.sample(pop, k)
    return min(selected, key=total_distance)

def cycle_crossover(p1,p2):
    size = len(p1)
    def create_child(parent1,parent2):
        child = [None]*size
        cycles_done = [False]*size
        index = 0
        while not all(cycles_done):
            while cycles_done[index]:
                index += 1
            start = index
            while True:
                child[index] = parent1[index]
                cycles_done[index] = True
                val = parent2[index]
                index = parent1.index(val)
                if index == start:
                    break
        for i in range(size):
            if child[i] is None:
                child[i] = parent2[i]
        return child
    return create_child(p1,p2), create_child(p2,p1)

def inversion_mutation(ind, rate=0.1):
    if random.random() < rate:
        i,j = sorted(random.sample(range(len(ind)),2))
        ind[i:j+1] = reversed(ind[i:j+1])
    return ind

def genetic_algo(nodes, pop_size=10, generations=50, mutation_rate=0.1):
    population = init_population(pop_size, nodes)
    best = min(population, key=total_distance)
    best_dist = total_distance(best)

    for _ in range(generations):
        new_pop = []
        while len(new_pop) < pop_size:
            p1 = tournament_selection(population)
            p2 = tournament_selection(population)
            c1,c2 = cycle_crossover(p1,p2)
            c1 = inversion_mutation(c1, mutation_rate)
            c2 = inversion_mutation(c2, mutation_rate)
            new_pop.extend([c1,c2])
        population = new_pop[:pop_size]
        current_best = min(population, key=total_distance)
        if total_distance(current_best)<best_dist:
            best_dist = total_distance(current_best)
            best = current_best
    return best, best_dist

# -------------------------
# Example Grid for A*
# 0=free, W=3, F=5
# -------------------------
grid = [
    [1,1,1,1,1,1],
    [1,1,3,1,5,1],
    [1,1,1,1,1,1],
    [1,5,1,1,1,1],
    [1,1,1,3,1,1],
    [1,1,1,1,1,1]
]

# Stage 1: A* S->P1->P2->G
start = (0,0)
P1 = (1,1)
P2 = (3,5)
goal = (5,5)

path1, cost1 = astar(grid, start, P1)
path2, cost2 = astar(grid, P1, P2)
path3, cost3 = astar(grid, P2, goal)

full_path = path1 + path2[1:] + path3[1:]
print("A* Full Path:", full_path)
print("A* Total Cost:", cost1+cost2+cost3)

# Stage 2: GA Delivery Route Optimization
nodes = list(customers.values())
best_route, best_dist = genetic_algo(nodes, pop_size=20, generations=100, mutation_rate=0.1)
print("\nGA Best Delivery Route:", best_route)
print("GA Best Total Distance:", best_dist)